# Dialogue Explorer

**Schema(s)** defining this data:
- `contracts/schemas/dialogue.schema.json` — **DialoguePack**: `dialogues[]` with dialogueId, sceneId, label, line, responseText, anchorVector, radius, effectVector, requiresRoomFeature, requiresItemTagPresent/Absent, requiresSkillId, takeItemTag, nextDialogueId. Narrative stat names in anchorVector/effectVector from STAT-AND-BEHAVIOUR-TAXONOMY. **Room feature** = room type only.

**Asset / pack** we load:
- `contracts/data/content_dialogue.json` — the dialogue pack

**What this tool does:** View options by scene; inspect narrative-stat anchors/radii and effects; filter by room feature or item tag; rebalance radius/effectVector; behaviour = requiresRoomFeature, requiresSkillId, requiresItemTag* assignments.

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "packages" / "engine").is_dir() and (ROOT.parent / "packages" / "engine").is_dir():
    ROOT = ROOT.parent
CONTRACTS_DIR = ROOT / "packages" / "engine" / "src" / "escape-the-dungeon" / "contracts"
SCHEMA_PATH = CONTRACTS_DIR / "schemas" / "dialogue.schema.json"
PACK_PATH = CONTRACTS_DIR / "data" / "content_dialogue.json"

with open(PACK_PATH, encoding="utf-8") as f:
    DIALOGUE_PACK = json.load(f)

print("Schema:", SCHEMA_PATH.relative_to(ROOT))
print("Pack:  ", PACK_PATH.relative_to(ROOT))
print(f"Loaded {len(DIALOGUE_PACK['dialogues'])} dialogue entries")

## View dialogues

Filter by sceneId or room feature. **Rebalance:** radius, effectVector. **Behaviour:** requiresRoomFeature (room type), requiresSkillId, requiresItemTagPresent/Absent.

In [ ]:
def view_dialogues(scene_id=None, room_feature=None):
    rows = DIALOGUE_PACK["dialogues"]
    if scene_id:
        rows = [r for r in rows if r.get("sceneId") == scene_id]
    if room_feature:
        rows = [r for r in rows if r.get("requiresRoomFeature") == room_feature]
    for d in rows:
        anchor = d.get("anchorVector") or {}
        effect = d.get("effectVector") or {}
        room = d.get("requiresRoomFeature", "—")
        skill = d.get("requiresSkillId", "—")
        tag_p = d.get("requiresItemTagPresent", "—")
        tag_a = d.get("requiresItemTagAbsent", "—")
        print(f"{d['dialogueId']:28} | scene={d.get('sceneId','')} | room={room} skill={skill} tagPresent={tag_p} tagAbsent={tag_a}")
        print(f"   label: {d.get('label','')}")
        print(f"   anchor={anchor} radius={d.get('radius')} effect={effect}")
    return rows

print("By sceneId:")
scenes = {d.get("sceneId") for d in DIALOGUE_PACK["dialogues"]}
print("Scenes:", sorted(scenes))
print("\nDialogues in treasure_scene:")
view_dialogues(scene_id="treasure_scene")
print("\nDialogues requiring room feature 'training':")
view_dialogues(room_feature="training")